# Question 23 - 28. Find the Index of the First Occurrence in a String

Given two strings `needle` and `haystack`, return the index of the first occurrence of `needle` in `haystack`, or `-1` if `needle` is not part of `haystack`.

**Example 1:**

    Input: haystack = "sadbutsad", needle = "sad"
    Output: 0
    Explanation: "sad" occurs at index 0 and 6.
    The first occurrence is at index 0, so we return 0.

**Example 2:**

    Input: haystack = "leetcode", needle = "leeto"
    Output: -1
    Explanation: "leeto" did not occur in "leetcode", so we return -1.

**Constraints:**

- `1 <= haystack.length, needle.length <= 10^4`
- `haystack` and `needle` consist of only lowercase English characters.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** String matching: sliding window brute force → KMP (prefix function / LPS array)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (check every start) | O(n · m) | O(1) |
| 2️⃣ Optimized (KMP) | O(n + m) | O(m) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"This is substring search. Let n = len(haystack) and m = len(needle). I'll ask whether built-ins like `str.find` are allowed. Usually the interviewer wants to see the algorithm."*

- Empty needle? → Not possible here (length ≥ 1). Conventionally the answer would be 0.
- `m > n` → can't match → `-1`.
- Lowercase only.
- **Built-in:** `haystack.find(needle)` is the one-liner. Say it, then implement the algorithm.

### Step 1 · Brute force

**Idea:** Try every start `i` from `0` to `n − m`. Compare `needle` against `haystack[i : i+m]` character by character, and stop at the first mismatch.

🗣️ *"The direct approach slides the needle across every position and compares. The worst case is O(n·m), for example haystack `aaaa…ab` and needle `aaab`: at every position we match almost the whole needle and then fail on the last character. With both up to 10⁴ that's 10⁸, which is too slow in theory. In practice, random text mismatches early, so brute force is often fine and a good first answer."*

In [1]:
class SolutionBrute:
    def strStr(self, haystack: str, needle: str) -> int:
        n, m = len(haystack), len(needle)
        for i in range(n - m + 1):
            j = 0
            while j < m and haystack[i + j] == needle[j]:
                j += 1
            if j == m:
                return i
        return -1

### Step 2 · Optimize

**Bottleneck:** after a mismatch, brute force shifts the needle by **one** and **forgets** everything it just matched. It then re-compares characters it has already seen.

**Key insight (KMP):** when I've matched `j` characters and then fail, I already know the last `j` characters of the haystack. They equal `needle[:j]`. The question is: *how much of the needle could still be lined up with those characters?* Answer: the **longest proper prefix of `needle[:j]` that is also a suffix** of it. Precompute this for every `j` once. It's called the **LPS** (longest prefix-suffix) array, or prefix function.

Then the haystack pointer **never moves backward**. On a mismatch we only fall back in the needle: `j = lps[j − 1]`.

🗣️ *"Brute force throws away information after a mismatch. KMP precomputes, for each prefix of the needle, the longest proper prefix that is also a suffix. When a mismatch happens after matching j characters, I can keep lps[j−1] characters matched instead of starting over, and the haystack index never goes backwards. Building the LPS array is O(m) and the scan is O(n), so O(n+m) total."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| `haystack.find(needle)` | ✅ in production | Optimised C. But the interview usually wants the algorithm. |
| Brute force sliding window | ⚠️ | O(n·m) worst case. Simple, and fine for small or random inputs. A good opening answer. |
| Rabin–Karp (rolling hash) | ✅ (alternative) | Expected O(n+m), O(1) space, but hash collisions mean you must verify matches. The worst case is O(n·m) with bad hashing. |
| **KMP (LPS array)** | ✅ (guaranteed) | Worst-case O(n+m) and deterministic. Costs O(m) for the LPS array. |
| Z-algorithm on `needle + '#' + haystack` | ✅ | Also O(n+m). Equivalent power, a different construction. |
| Boyer–Moore | ⚠️ | Very fast in practice for large alphabets, but complex to code correctly in an interview. |

In [2]:
class Solution:
    def strStr(self, haystack: str, needle: str) -> int:
        n, m = len(haystack), len(needle)
        if m == 0:
            return 0

        # lps[i] = length of the longest proper prefix of needle[:i+1]
        #          that is also a suffix of it
        lps = [0] * m
        length = 0                          # current matched prefix length
        for i in range(1, m):
            while length > 0 and needle[i] != needle[length]:
                length = lps[length - 1]    # fall back to a shorter border
            if needle[i] == needle[length]:
                length += 1
            lps[i] = length

        # scan haystack; i never moves backward
        j = 0                               # chars of needle matched so far
        for i in range(n):
            while j > 0 and haystack[i] != needle[j]:
                j = lps[j - 1]              # reuse the overlap instead of restarting
            if haystack[i] == needle[j]:
                j += 1
            if j == m:
                return i - m + 1
        return -1

### Step 3 · Toy example walkthrough

**Build LPS for `needle = "ababc"`**

| i | needle[:i+1] | longest prefix = suffix | lps[i] |
|---|---|---|---|
| 0 | a | n/a (proper, so not the whole thing) | 0 |
| 1 | ab | none | 0 |
| 2 | ab**a** | "a" | 1 |
| 3 | ab**ab** | "ab" | 2 |
| 4 | ababc | none | 0 |

`lps = [0, 0, 1, 2, 0]`

**Search in `haystack = "abababc"`**

| i | haystack[i] | j before | compare | action | j after |
|---|---|---|---|---|---|
| 0 | a | 0 | a = a | match | 1 |
| 1 | b | 1 | b = b | match | 2 |
| 2 | a | 2 | a = a | match | 3 |
| 3 | b | 3 | b = b | match | 4 |
| 4 | a | 4 | a ≠ c | **fall back** j = lps[3] = 2; now a = needle[2] = a | 3 |
| 5 | b | 3 | b = b | match | 4 |
| 6 | c | 4 | c = c | match | **5 = m** → return 6 − 5 + 1 = **2** ✅ |

At i = 4, brute force would restart at haystack index 1 and recheck "bab…". KMP knows the last 4 characters "abab" end in "ab", which is the start of the needle, so it continues with 2 characters already matched. The haystack index never moved back.

In [3]:
cases = [
    ("sadbutsad", "sad", 0),
    ("leetcode", "leeto", -1),
    ("abababc", "ababc", 2),
    ("a", "a", 0),
    ("abc", "c", 2),
    ("aaa", "aaaa", -1),
    ("aaaaab", "aab", 3),
    ("mississippi", "issip", 4),
]
for h, nd, expected in cases:
    assert SolutionBrute().strStr(h, nd) == expected
    assert Solution().strStr(h, nd) == expected
    assert h.find(nd) == expected
    print(h, nd, "->", expected)

import random
for _ in range(2000):
    h = "".join(random.choices("ab", k=random.randint(1, 15)))
    nd = "".join(random.choices("ab", k=random.randint(1, 4)))
    assert Solution().strStr(h, nd) == h.find(nd), (h, nd)
print("All tests passed ✅")

sadbutsad sad -> 0
leetcode leeto -> -1
abababc ababc -> 2
a a -> 0
abc c -> 2
aaa aaaa -> -1
aaaaab aab -> 3
mississippi issip -> 4
All tests passed ✅


### Step 4 · Wrap up

| | Time | Space | Why |
|---|---|---|---|
| Brute force | O(n · m) worst | O(1) | Restarts after each mismatch |
| Rabin–Karp | O(n + m) expected | O(1) | Rolling hash plus verification |
| **KMP** | **O(n + m)** worst | **O(m)** | LPS reuse, and the haystack index never backs up |

**Complexity, explained (the amortised argument):**
- In the search loop, `j` increases by at most 1 per character of the haystack, so it increases at most n times in total. Each fallback `j = lps[j−1]` *decreases* `j`, and it can't go below 0. **You can't decrease more than you've increased**, so the total number of fallbacks is ≤ n. The search is O(n).
- The same argument on the needle makes the LPS build O(m).
- The `lps` array is O(m) space.

**Edge cases:** needle longer than haystack, match at the very end, overlapping partial matches (`"aaaaab"`/`"aab"`), single characters.

**Follow-ups:**
- *Find **all** occurrences?* → On a full match, record it and set `j = lps[j−1]` to continue.
- *Repeated Substring Pattern (LC 459)* → `lps[-1]` tells you the smallest period.

🗣️ **30-second recap:** *"Brute force tries every start and compares, which is O(n·m) in the worst case. KMP precomputes, for each needle prefix, the longest proper prefix that's also a suffix. On a mismatch I fall back to that length instead of restarting, so the haystack pointer only moves forward. Amortised, that's O(n+m) time with O(m) extra space."*